In [3]:
!pip install librosa

In [3]:
import os
import random
import librosa
import numpy as np
import librosa
import pandas as pd
import matplotlib.pyplot as plt

BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

STEMS_PATH = os.path.join(BASE_PATH, "genres_stems")
MASHUPS_PATH = os.path.join(BASE_PATH, "mashups")
TEST_CSV = os.path.join(BASE_PATH, "test.csv")
ESC_AUDIO = os.path.join(BASE_PATH, "ESC-50-master/audio")

GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]



In [5]:
#test data size:
test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")
print("Number of test mashups:", len(test_df))
test_df.head()

Number of test mashups: 3020


,id,filename
0,1,mashups/song0001.wav
1,2,mashups/song0002.wav
2,3,mashups/song0003.wav
3,4,mashups/song0004.wav
4,5,mashups/song0005.wav


In [10]:
!pip install --upgrade wandb --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 60.8 MB/s eta 0:00:00:00:0100:01


In [18]:

# import wandb

# wandb.init(
#     project="milestone2",
#     name="mfcc-classical-ml",
#     config={
#         "features": "MFCC",
#         "models": ["LogisticRegression", "NaiveBayes"],
#         "metric": "MacroF1"
#     }
# )

AttributeError: module 'wandb.sdk' has no attribute 'lib'

In [16]:
# !wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 


Aborted!


## EDA

In [ ]:
#Class distribution (songs per genre)
genre_counts = {}

for g in GENRES:
    genre_dir = os.path.join(STEMS_PATH, g)
    songs = [
        d for d in os.listdir(genre_dir)
        if os.path.isdir(os.path.join(genre_dir, d))
    ]
    genre_counts[g] = len(songs)

df_genres = pd.DataFrame.from_dict(
    genre_counts, orient="index", columns=["num_songs"]
)

df_genres


In [ ]:
#Stem availability / gaps
stem_files = ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]
stem_stats = []

for g in GENRES:
    genre_dir = os.path.join(STEMS_PATH, g)
    songs = [
        d for d in os.listdir(genre_dir)
        if os.path.isdir(os.path.join(genre_dir, d))
    ]

    for s in songs:
        song_dir = os.path.join(genre_dir, s)
        available = sum(
            os.path.exists(os.path.join(song_dir, stem))
            for stem in stem_files
        )
        stem_stats.append({
            "genre": g,
            "song": s,
            "num_stems": available
        })

df_stems = pd.DataFrame(stem_stats)
df_stems["num_stems"].value_counts().sort_index()



In [ ]:
#Audio length analysis — training stems
durations = []

for g in GENRES:
    genre_dir = os.path.join(STEMS_PATH, g)
    songs = [
        d for d in os.listdir(genre_dir)
        if os.path.isdir(os.path.join(genre_dir, d))
    ]

    for s in random.sample(songs, 5):  # sample few per genre
        song_dir = os.path.join(genre_dir, s)
        for stem in stem_files:
            stem_path = os.path.join(song_dir, stem)
            if os.path.exists(stem_path):
                audio, sr = librosa.load(stem_path, sr=None)
                durations.append(len(audio) / sr)

pd.Series(durations).describe()


In [ ]:
#Mashup duration stats
test_lengths = []

for fname in test_df["filename"].sample(20, random_state=42):
    path = os.path.join(BASE_PATH, fname)
    audio, sr = librosa.load(path, sr=None)
    test_lengths.append(len(audio) / sr)

pd.Series(test_lengths).describe()


In [ ]:
#esc 50
noise_file = random.choice(os.listdir(ESC_AUDIO))
noise_path = os.path.join(ESC_AUDIO, noise_file)

noise, sr = librosa.load(noise_path, sr=22050)
print("Noise file:", noise_file)
print("Duration:", len(noise)/sr)

noise_spec = librosa.feature.melspectrogram(y=noise, sr=sr)
noise_log = librosa.power_to_db(noise_spec)

plt.figure(figsize=(10, 3))
plt.imshow(noise_log, aspect="auto", origin="lower")
plt.title("ESC-50 Noise Spectrogram")
plt.colorbar()
plt.show()


## model building

In [ ]:
!pip install numpy==1.26.4 librosa --quiet

In [10]:
import os
import random
import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# ================= PATHS ================= #

BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

STEMS_PATH = os.path.join(BASE_PATH, "genres_stems")
MASHUPS_PATH = os.path.join(BASE_PATH, "mashups")
TEST_CSV = os.path.join(BASE_PATH, "test.csv")
ESC_AUDIO = os.path.join(BASE_PATH, "ESC-50-master/audio")

GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]

# ================= CONFIG ================= #

SR = 16000
DURATION = 5
SAMPLES = SR * DURATION
MIN_STEMS_REQUIRED = 2

random.seed(42)
np.random.seed(42)

# ================= AUDIO ================= #

def load_audio(path):
    audio, _ = librosa.load(path, sr=SR, mono=True)
    if len(audio) < SAMPLES:
        audio = np.pad(audio, (0, SAMPLES - len(audio)))
    return audio[:SAMPLES]

def extract_mel(audio):
    mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=64)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db.astype(np.float32)

# ================= BUILD DATA ================= #

X_train, y_train = [], []

print("Building dataset...")

for genre in GENRES:
    genre_path = os.path.join(STEMS_PATH, genre)

    song_dirs = [
        d for d in os.listdir(genre_path)
        if os.path.isdir(os.path.join(genre_path, d))
    ]

    for _ in range(100):
        chosen = random.sample(song_dirs, 4)
        mix = np.zeros(SAMPLES)
        used = 0

        for stem_name, song in zip(
            ["drums.wav", "bass.wav", "vocals.wav", "others.wav"],
            chosen
        ):
            stem_path = os.path.join(genre_path, song, stem_name)
            if not os.path.exists(stem_path):
                continue

            mix += load_audio(stem_path)
            used += 1

        if used < MIN_STEMS_REQUIRED:
            continue

        mix = mix / (np.max(np.abs(mix)) + 1e-6)

        X_train.append(extract_mel(mix))
        y_train.append(genre)

X_train = np.array(X_train)
y_train = np.array(y_train)

print("Total samples:", len(X_train))

# ================= LABEL ENCODING ================= #

label_map = {g: i for i, g in enumerate(GENRES)}
inv_label_map = {i: g for g, i in label_map.items()}

# ================= DATASET ================= #

class AudioDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        x = np.expand_dims(x, axis=0)  # (1, H, W)
        y = label_map[self.y[idx]]
        return torch.tensor(x), torch.tensor(y)

# ================= SPLIT ================= #

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

train_loader = DataLoader(AudioDataset(X_tr, y_tr), batch_size=16, shuffle=True)
val_loader = DataLoader(AudioDataset(X_val, y_val), batch_size=16)

# ================= MODEL ================= #

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.pool = nn.AdaptiveAvgPool2d((4, 4))

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.pool(x)
        return self.fc(x)

# ================= TRAIN SETUP ================= #

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ================= TRAIN ================= #

def train_epoch():
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

# ================= EVAL ================= #

def evaluate():
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            out = model(x)

            p = torch.argmax(out, dim=1).cpu().numpy()
            preds.extend(p)
            targets.extend(y.numpy())

    return f1_score(targets, preds, average="macro")

# ================= RUN ================= #

EPOCHS = 5
"""
for epoch in range(EPOCHS):
    loss = train_epoch()
    f1 = evaluate()

    wandb.log({"epoch": epoch, "loss": loss, "val_f1": f1})
    print(f"Epoch {epoch} | Loss: {loss:.4f} | F1: {f1:.4f}")

wandb.finish()"""

Building dataset...


KeyboardInterrupt: 

In [7]:
# ================= TEST INFERENCE ================= #

print("Running CNN inference on test data...")

test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")

model.eval()
predictions = []

with torch.no_grad():
    for fname in test_df["filename"]:
        audio_path = os.path.join(BASE_PATH, fname)

        audio = load_audio(audio_path)
        mel = extract_mel(audio)

        x = np.expand_dims(mel, axis=0)   # (1, H, W)
        x = np.expand_dims(x, axis=0)     # (1, 1, H, W)

        x = torch.tensor(x).to(device)

        out = model(x)
        pred = torch.argmax(out, dim=1).item()

        predictions.append(inv_label_map[pred])


# ================= SUBMISSION ================= #

submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})

OUT_PATH = "/kaggle/working/submission.csv"
submission.to_csv(OUT_PATH, index=False)

print("Submission saved at:", OUT_PATH)
print(submission.head())

Running CNN inference on test data...
Submission saved at: /kaggle/working/submission.csv
   id  genre
0   1  metal
1   2  metal
2   3  metal
3   4  metal
4   5  metal


In [ ]:
# wandb.finish()

In [ ]:
#milestone 5

In [8]:
from transformers import HubertModel, Wav2Vec2FeatureExtractor
import torch.nn as nn
import torch

2026-03-18 18:04:23.786332: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773857063.981098      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773857064.039491      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773857064.518062      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773857064.518097      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773857064.518099      55 computation_placer.cc:177] computation placer alr

In [12]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    "facebook/hubert-base-ls960"
)
class HubertDataset(torch.utils.data.Dataset):
    def __init__(self, X_audio, y_labels):
        self.X = X_audio
        self.y = y_labels

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        audio = self.X[idx]

        inputs = feature_extractor(
            audio,
            sampling_rate=SR,
            return_tensors="pt",
            padding=True
        )

        x = inputs.input_values.squeeze(0)
        y = label_map[self.y[idx]]

        return x, torch.tensor(y)
X_audio, y_audio = [], []

print("Building raw audio dataset for transformer...")

for genre in GENRES:
    genre_path = os.path.join(STEMS_PATH, genre)

    song_dirs = [
        d for d in os.listdir(genre_path)
        if os.path.isdir(os.path.join(genre_path, d))
    ]

    for _ in range(50):  # keep smaller (transformer heavy)
        chosen = random.sample(song_dirs, 4)
        mix = np.zeros(SAMPLES)
        used = 0

        for stem_name, song in zip(
            ["drums.wav", "bass.wav", "vocals.wav", "others.wav"],
            chosen
        ):
            stem_path = os.path.join(genre_path, song, stem_name)
            if not os.path.exists(stem_path):
                continue

            mix += load_audio(stem_path)
            used += 1

        if used < MIN_STEMS_REQUIRED:
            continue

        mix = mix / (np.max(np.abs(mix)) + 1e-6)

        X_audio.append(mix.astype(np.float32))
        y_audio.append(genre)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_audio, y_audio, test_size=0.2, stratify=y_audio, random_state=42
)

train_ds = HubertDataset(X_tr, y_tr)
val_ds = HubertDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4)
class HubertClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.hubert = HubertModel.from_pretrained("facebook/hubert-base-ls960")
        self.classifier = nn.Linear(self.hubert.config.hidden_size, num_classes)

    def forward(self, x):
        outputs = self.hubert(x)
        hidden = outputs.last_hidden_state

        pooled = hidden.mean(dim=1)
        return self.classifier(pooled)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HubertClassifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
def train_epoch():
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

def evaluate():
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            out = model(x)

            p = torch.argmax(out, dim=1).cpu().numpy()
            preds.extend(p)
            targets.extend(y.numpy())

    return f1_score(targets, preds, average="macro")
EPOCHS = 3

#for epoch in range(EPOCHS):
    #loss = train_epoch()
    #f1 = evaluate()

    #wandb.log({"epoch": epoch, "loss": loss, "val_f1": f1})
    #print(f"[HuBERT] Epoch {epoch} | Loss: {loss:.4f} | F1: {f1:.4f}")

Building raw audio dataset for transformer...
